In [22]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [2]:
DAX = pd.read_csv("data/DAX.csv")
EURtoGBP = pd.read_csv("data/EURtoGBP.csv")
EURtoUSD = pd.read_csv("data/EURtoUSD.csv")
close_prices = pd.read_csv("data/close_prices.csv")
interest_rates = pd.read_csv("data/interest_rates.csv")
monthly_close_prices = pd.read_csv("data/monthly_close_prices.csv")
monthly_returns = pd.read_csv("data/monthly_returns.csv")
monthly_volumes = pd.read_csv("data/monthly_volumes.csv")
returns = pd.read_csv("data/returns.csv")
unemployment = pd.read_csv("data/unemployment.csv")
volumes = pd.read_csv("data/volumes.csv")
merged_daily_market = pd.read_csv("data/merged_daily_market.csv")
merged_monthly_market = pd.read_csv("data/merged_monthly_market.csv")

## Pytorch LSTM Model

In [28]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Create directory for models
os.makedirs('models', exist_ok=True)

# =====================================
# Helper Functions
# =====================================
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim=50, num_layers=2):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])
        return out

def create_sequences(data, seq_length=10, dates=None):
    X, y, y_dates, y_indices = [], [], [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length, 0])  # predict close
        if dates is not None:
            y_dates.append(dates[i+seq_length])
            y_indices.append(i+seq_length)
    return np.array(X), np.array(y), np.array(y_dates), np.array(y_indices)

def inverse_transform_predictions(scaled_preds, scaler, feature_index=0):
    dummy = np.zeros((len(scaled_preds), scaler.n_features_in_))
    dummy[:, feature_index] = scaled_preds
    inverted = scaler.inverse_transform(dummy)
    return inverted[:, feature_index]

def save_complete_model(ticker, model, scaler, df_t, X, test_mae, train_mae):
    """Save everything needed for future predictions"""
    model_assets = {
        'model_state_dict': model.state_dict(),
        'model_config': {
            'input_dim': X.shape[2],
            'hidden_dim': 50,
            'num_layers': 2
        },
        'scaler': scaler,
        'seq_length': 10,
        'feature_columns': ['close', 'volume', '1eur_usd', '1eur_gbp'],
        'ticker': ticker,
        'last_training_date': df_t['date'].max().strftime('%Y-%m-%d'),
        'last_sequence': scaler.transform(df_t[['close', 'volume', '1eur_usd', '1eur_gbp']].tail(10)),
        'min_max_values': {
            'close_min': df_t['close'].min(),
            'close_max': df_t['close'].max()
        },
        'performance_metrics': {
            'test_mae': test_mae,
            'train_mae': train_mae,
            'test_rmse': np.sqrt(mean_squared_error(test_actual_prices, test_preds_inv))
        }
    }
    
    torch.save(model_assets, f'models/lstm_model_{ticker.replace(".", "_")}_complete.pth')
    return model_assets

# =====================================
# Main Training Loop
# =====================================
def train_models_for_all_tickers(merged_daily_market, max_tickers=None):
    """Train LSTM models for all tickers in the dataset"""
    
    all_tickers = sorted(list(set(merged_daily_market['ticker'])))
    if max_tickers:
        all_tickers = all_tickers[:max_tickers]  # For testing with fewer tickers
    
    results = []
    
    for i, ticker in enumerate(all_tickers):
        print(f"\n{'='*60}")
        print(f"Training model {i+1}/{len(all_tickers)}: {ticker}")
        print(f"{'='*60}")
        
        try:
            # =====================================
            # 1️⃣ Prepare Data
            # =====================================
            df_t = merged_daily_market[merged_daily_market['ticker'] == ticker].copy()
            
            # Check if we have enough data
            if len(df_t) < 50:
                print(f"⚠️  Skipping {ticker}: insufficient data ({len(df_t)} rows)")
                continue
                
            df_t = df_t[['date', 'close', 'volume', '1eur_usd', '1eur_gbp']].dropna()
            df_t['date'] = pd.to_datetime(df_t['date'])
            df_t.sort_values('date', inplace=True)
            
            dates = df_t['date'].values
            close_prices = df_t['close'].values

            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(df_t[['close', 'volume', '1eur_usd', '1eur_gbp']])
            
            # =====================================
            # 2️⃣ Create Sequences
            # =====================================
            X, y, y_dates, y_indices = create_sequences(scaled, seq_length=10, dates=dates)
            
            if len(X) < 20:
                print(f"⚠️  Skipping {ticker}: not enough sequences after processing")
                continue
                
            split = int(len(X) * 0.8)
            X_train, y_train = X[:split], y[:split]
            X_test, y_test, test_dates, test_indices = X[split:], y[split:], y_dates[split:], y_indices[split:]
            
            # =====================================
            # 3️⃣ Create DataLoaders
            # =====================================
            train_loader = DataLoader(TimeSeriesDataset(X_train, y_train), batch_size=32, shuffle=True)
            test_loader = DataLoader(TimeSeriesDataset(X_test, y_test), batch_size=32, shuffle=False)
            
            # =====================================
            # 4️⃣ Initialize Model
            # =====================================
            model = LSTMModel(input_dim=X.shape[2])
            criterion = nn.MSELoss()
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            
            # =====================================
            # 5️⃣ Train Model
            # =====================================
            epochs = 50
            train_losses = []
            
            for epoch in range(epochs):
                model.train()
                total_loss = 0
                for X_batch, y_batch in train_loader:
                    optimizer.zero_grad()
                    preds = model(X_batch).squeeze()
                    loss = criterion(preds, y_batch)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()
                    total_loss += loss.item()
                
                avg_loss = total_loss/len(train_loader)
                train_losses.append(avg_loss)
                if (epoch+1) % 20 == 0:
                    print(f"  Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.6f}")
            
            # =====================================
            # 6️⃣ Evaluate Model
            # =====================================
            model.eval()
            train_preds, test_preds = [], []
            
            with torch.no_grad():
                for X_batch, _ in train_loader:
                    train_preds.extend(model(X_batch).squeeze().tolist())
                for X_batch, _ in test_loader:
                    test_preds.extend(model(X_batch).squeeze().tolist())
            
            # Inverse transform
            train_preds_inv = inverse_transform_predictions(train_preds, scaler, 0)
            test_preds_inv = inverse_transform_predictions(test_preds, scaler, 0)
            
            # Get actual prices
            train_actual_prices = close_prices[y_indices[:split]]
            test_actual_prices = close_prices[test_indices]
            
            # Calculate metrics
            train_mae = mean_absolute_error(train_actual_prices, train_preds_inv)
            test_mae = mean_absolute_error(test_actual_prices, test_preds_inv)
            test_rmse = np.sqrt(mean_squared_error(test_actual_prices, test_preds_inv))
            
            # =====================================
            # 7️⃣ Save Model
            # =====================================
            model_assets = save_complete_model(ticker, model, scaler, df_t, X, test_mae, train_mae)
            
            # =====================================
            # 8️⃣ Store Results
            # =====================================
            result = {
                'ticker': ticker,
                'train_mae': train_mae,
                'test_mae': test_mae,
                'test_rmse': test_rmse,
                'training_samples': len(X_train),
                'test_samples': len(X_test),
                'total_samples': len(df_t),
                'status': 'SUCCESS'
            }
            results.append(result)
            
            print(f"✅ {ticker} - Train MAE: {train_mae:.2f}, Test MAE: {test_mae:.2f}, Test RMSE: {test_rmse:.2f}")
            
            # Create directory for plots
            os.makedirs('model_plots', exist_ok=True)
            
            # 9️⃣ Save plots for ALL models
            plt.figure(figsize=(12, 6))
            
            # Plot 1: Full timeline with train/test split
            plt.subplot(2, 1, 1)
            plt.plot(df_t['date'], df_t['close'], label='Actual', color='blue', alpha=0.7)
            train_dates = y_dates[:split]
            plt.plot(train_dates, train_preds_inv, label='Train Predicted', color='green', alpha=0.8)
            plt.plot(test_dates, test_preds_inv, label='Test Predicted', color='red', alpha=0.8)
            
            split_date = test_dates[0]
            plt.axvline(x=split_date, color='black', linestyle='--', alpha=0.7, label='Train/Test Split')
            plt.title(f"LSTM Forecast - {ticker} (Test MAE: {test_mae:.2f})")
            plt.ylabel("Close Price (EUR)")
            plt.legend()
            plt.xticks(rotation=45)
            
            # Plot 2: Zoom in on test period
            plt.subplot(2, 1, 2)
            plt.plot(test_dates, test_actual_prices, label='Actual', color='blue', linewidth=2)
            plt.plot(test_dates, test_preds_inv, label='Predicted', color='red', linewidth=2)
            plt.title(f"Test Period - {ticker}")
            plt.xlabel("Date")
            plt.ylabel("Close Price (EUR)")
            plt.legend()
            plt.xticks(rotation=45)
            
            plt.tight_layout()
            plt.savefig(f'model_plots/{ticker.replace(".", "_")}_prediction.png', dpi=150, bbox_inches='tight')
            plt.close()  # Important: close the figure to free memory
            
            print(f"📊 Plot saved: model_plots/{ticker.replace('.', '_')}_prediction.png")
                        
        except Exception as e:
            print(f"❌ Error training {ticker}: {str(e)}")
            results.append({
                'ticker': ticker,
                'status': 'FAILED',
                'error': str(e)
            })
            continue
    
    # =====================================
    # 🔟 Summary Report
    # =====================================
    print(f"\n{'='*60}")
    print("TRAINING SUMMARY")
    print(f"{'='*60}")
    
    successful = [r for r in results if r['status'] == 'SUCCESS']
    failed = [r for r in results if r['status'] == 'FAILED']
    
    print(f"✅ Successful: {len(successful)} models")
    print(f"❌ Failed: {len(failed)} models")
    
    if successful:
        avg_test_mae = np.mean([r['test_mae'] for r in successful])
        best_model = min(successful, key=lambda x: x['test_mae'])
        worst_model = max(successful, key=lambda x: x['test_mae'])
        
        print(f"\n📊 Average Test MAE: {avg_test_mae:.2f}")
        print(f"🏆 Best Model: {best_model['ticker']} (MAE: {best_model['test_mae']:.2f})")
        print(f"📉 Worst Model: {worst_model['ticker']} (MAE: {worst_model['test_mae']:.2f})")
    
    # Save results to CSV
    results_df = pd.DataFrame(results)
    results_df.to_csv('model_training_results.csv', index=False)
    print(f"\n📄 Results saved to 'model_training_results.csv'")
    
    return results

# =====================================
# 🚀 Run the Training
# =====================================
if __name__ == "__main__":
    # For testing, you might want to limit the number of tickers first
    results = train_models_for_all_tickers(
        merged_daily_market
    )


Training model 1/40: ADS.DE
  Epoch 20/50 - Loss: 0.004707
  Epoch 40/50 - Loss: 0.004288
✅ ADS.DE - Train MAE: 25.91, Test MAE: 4.31, Test RMSE: 6.08
📊 Plot saved: model_plots/ADS_DE_prediction.png

Training model 2/40: AIR.PA
  Epoch 20/50 - Loss: 0.003276
  Epoch 40/50 - Loss: 0.002080
✅ AIR.PA - Train MAE: 14.73, Test MAE: 4.90, Test RMSE: 5.66
📊 Plot saved: model_plots/AIR_PA_prediction.png

Training model 3/40: ALV.DE
  Epoch 20/50 - Loss: 0.001794
  Epoch 40/50 - Loss: 0.000952
✅ ALV.DE - Train MAE: 43.78, Test MAE: 6.22, Test RMSE: 7.66
📊 Plot saved: model_plots/ALV_DE_prediction.png

Training model 4/40: BAS.DE
  Epoch 20/50 - Loss: 0.005316
  Epoch 40/50 - Loss: 0.004590
✅ BAS.DE - Train MAE: 3.27, Test MAE: 0.78, Test RMSE: 1.04
📊 Plot saved: model_plots/BAS_DE_prediction.png

Training model 5/40: BAYN.DE
  Epoch 20/50 - Loss: 0.002531
  Epoch 40/50 - Loss: 0.001771
✅ BAYN.DE - Train MAE: 5.50, Test MAE: 1.01, Test RMSE: 1.21
📊 Plot saved: model_plots/BAYN_DE_prediction.png